In [1]:
import json
import logging
import glob
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger(__name__)

# Configuration
EPSS_THRESHOLD = 0.95
DATA_DIR = Path('data')
INPUT_FILES = {
    'metasploit': 'metasploit.txt',
    'nuclei': 'nuclei.txt',
    'cisa': 'known_exploited_vulnerabilities.csv',
    'epss': 'epss_scores-current.csv',
    'nvd_pattern': 'nvd.jsonl'
}

In [2]:
def load_cve_list(filepath, source_name):
    """Load CVE list from a text file with error handling."""
    if not Path(filepath).exists():
        logger.warning(f"{filepath} not found, skipping {source_name}")
        return pd.DataFrame(columns=['CVE', 'Source'])
    
    try:
        df = pd.read_csv(filepath, header=None, names=['CVE'])
        df.drop_duplicates(inplace=True)
        df['Source'] = source_name
        return df[['CVE', 'Source']]
    except Exception as e:
        logger.error(f"Error loading {filepath}: {e}")
        return pd.DataFrame(columns=['CVE', 'Source'])

metasploit_df = load_cve_list(INPUT_FILES['metasploit'], 'Metasploit')
nuclei_df = load_cve_list(INPUT_FILES['nuclei'], 'Nuclei')

In [3]:
def load_cisa_data(filepath):
    """Load CISA Known Exploited Vulnerabilities data."""
    if not Path(filepath).exists():
        logger.warning(f"{filepath} not found, skipping CISA data")
        return pd.DataFrame(columns=['CVE', 'Source'])
    
    try:
        df = pd.read_csv(filepath)
        df = df.rename(columns={"cveID": "CVE"})
        df['Source'] = 'CISA'
        return df[['CVE', 'Source']]
    except Exception as e:
        logger.error(f"Error loading CISA data: {e}")
        return pd.DataFrame(columns=['CVE', 'Source'])

CISA_df = load_cisa_data(INPUT_FILES['cisa'])

In [4]:
def load_epss_data(filepath, threshold=EPSS_THRESHOLD):
    """Load EPSS data and filter by threshold."""
    if not Path(filepath).exists():
        logger.warning(f"{filepath} not found, skipping EPSS data")
        return pd.DataFrame(columns=['CVE', 'Source']), pd.DataFrame()
    
    try:
        df = pd.read_csv(filepath, skiprows=1)
        df = df.rename(columns={"cve": "CVE"})
        df_all = df.copy()
        df_filtered = df[df.epss > threshold].copy()
        df_filtered['Source'] = 'EPSS'
        return df_filtered[['CVE', 'Source']], df_all
    except Exception as e:
        logger.error(f"Error loading EPSS data: {e}")
        return pd.DataFrame(columns=['CVE', 'Source']), pd.DataFrame()

epss_df, epss_df_all = load_epss_data(INPUT_FILES['epss'])

In [5]:
def merge_cve_sources(dfs):
    """Merge multiple CVE source dataframes and combine sources."""
    if not dfs:
        logger.warning("No dataframes to merge")
        return pd.DataFrame(columns=['CVE', 'Source'])
    
    merged = pd.concat(dfs, ignore_index=True, sort=False)
    merged = merged.groupby('CVE', as_index=False).agg({'Source': '/'.join})
    return merged

CVE_list = merge_cve_sources([epss_df, CISA_df, metasploit_df, nuclei_df])

In [6]:
def safe_get(d, keys, default='Missing_Data'):
    """Safely get nested dictionary values."""
    for key in keys:
        try:
            d = d[key]
        except (KeyError, TypeError, IndexError):
            return default
    return d

def parse_nvd_entry(entry):
    """Parse a single NVD entry and extract relevant fields."""
    cve = entry['cve']['id']
    
    # Extract CVSS v3.1 metrics with safe navigation
    metrics = safe_get(entry, ['cve', 'metrics', 'cvssMetricV31', 0], {})
    cvss_data = safe_get(metrics, ['cvssData'], {})
    
    published_date = safe_get(entry, ['cve', 'published'], 'Missing_Data')
    base_score = safe_get(cvss_data, ['baseScore'], '0.0')
    description = safe_get(entry, ['cve', 'descriptions', 0, 'value'], '')
    
    # Skip rejected/disputed CVEs
    if description.startswith('** REJECT **'):
        return None
    
    return {
        'CVE': cve,
        'Published': published_date,
        'CVSS Score': base_score,
        'Description': description
    }

def load_nvd_data(pattern):
    """Load and parse NVD JSONL data files."""
    row_accumulator = []
    files = glob.glob(pattern)
    
    if not files:
        logger.warning(f"No NVD files found matching pattern: {pattern}")
        return pd.DataFrame(columns=['CVE', 'Published', 'CVSS Score', 'Description'])
    
    for filename in files:
        try:
            with open(filename, 'r', encoding='utf-8') as f:
                nvd_data = json.load(f)
                for entry in nvd_data:
                    parsed = parse_nvd_entry(entry)
                    if parsed:
                        row_accumulator.append(parsed)
        except Exception as e:
            logger.error(f"Error parsing {filename}: {e}")
    
    if not row_accumulator:
        logger.warning("No valid NVD entries found")
        return pd.DataFrame(columns=['CVE', 'Published', 'CVSS Score', 'Description'])
    
    nvd = pd.DataFrame(row_accumulator)
    nvd['Published'] = pd.to_datetime(nvd['Published'])
    nvd = nvd.sort_values(by=['Published']).reset_index(drop=True)
    return nvd

nvd = load_nvd_data(INPUT_FILES['nvd_pattern'])

In [7]:
def create_final_output(cve_list, nvd_data, epss_all, output_path):
    """Merge all data sources and create final output."""
    if cve_list.empty or nvd_data.empty:
        logger.error("Cannot merge: CVE list or NVD data is empty")
        return
    
    try:
        # Merge CVE list with NVD data
        merged = pd.merge(cve_list, nvd_data, how='inner', left_on='CVE', right_on='CVE')
        
        # Merge with EPSS data
        merged = pd.merge(merged, epss_all, how='inner', left_on='CVE', right_on='CVE')
        
        # Select and rename columns
        result = merged[['CVE', 'CVSS Score', 'epss', 'Description', 'Published', 'Source']]
        result = result.rename(columns={"epss": "EPSS"})
        
        # Ensure output directory exists
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Save to CSV
        result.to_csv(output_path, index=False)
        logger.info(f"Successfully saved {len(result)} CVEs to {output_path}")
        
        return result
    except Exception as e:
        logger.error(f"Error creating final output: {e}")
        return None

patchthisapp_df = create_final_output(
    CVE_list, 
    nvd, 
    epss_df_all, 
    DATA_DIR / 'data.csv'
)

INFO: Successfully saved 7728 CVEs to data/data.csv
